# TODO 4 — Analyse des paramètres du Simulated Annealing

Ce notebook étudie l'impact des différents paramètres du recuit simulé sur :
- la **qualité** de la solution obtenue (valeur QUBO)
- la **stabilité** des résultats (variance entre runs)
- le **temps de calcul**

Paramètres étudiés :
1. Température initiale `initial_temperature` (T₀)
2. Facteur de décroissance `cooling_rate` (β)
3. Seuil d'arrêt `min_temperature` (T_min)
4. Nombre de pas par palier de température `steps_per_temperature`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import time

from problem.qubo import QuboProblem
from solver.classical_solver.simulated_annealing import SimulatedAnnealing
from utils.display_methods import plot_convergence_runs, plot_parameter_sweep, plot_boxplot_comparison

np.random.seed(0)
print('Imports OK')

## 1. Création d'une instance QUBO difficile

On génère une instance aléatoire de taille n=20 avec des coefficients entre -10 et +10.
Une taille de 20 variables donne un espace de recherche de 2²⁰ ≈ 1 million de solutions,
ce qui est suffisamment grand pour que SA ne converge pas systématiquement.

In [ ]:
N = 20  # taille de l'instance
rng = np.random.RandomState(42)
Q_matrix = np.triu(rng.randint(-10, 10, size=(N, N)).astype(float))
qubo = QuboProblem(Q_matrix)

print(f'Instance QUBO : n={qubo.n}, espace de recherche = 2^{qubo.n} = {2**qubo.n:,} solutions')

## 2. Non-convergence avec les paramètres par défaut

On exécute SA 30 fois avec les paramètres par défaut et on observe la dispersion des résultats.
Si l'algorithme avait convergé, toutes les runs donneraient le même résultat optimal.
La variance observée indique que l'heuristique n'a pas convergé de façon fiable.

In [ ]:
N_RUNS = 30
default_params = dict(initial_temperature=100.0, cooling_rate=0.95, min_temperature=0.01, steps_per_temperature=1)

results_default = []
for _ in range(N_RUNS):
    sa = SimulatedAnnealing(qubo, maximize=False, **default_params)
    val, _ = sa.solve()
    results_default.append(val)

print(f'Paramètres : {default_params}')
print(f'Résultats sur {N_RUNS} runs :')
print(f'  Min    : {np.min(results_default):.2f}')
print(f'  Max    : {np.max(results_default):.2f}')
print(f'  Moyenne: {np.mean(results_default):.2f}')
print(f'  Std    : {np.std(results_default):.2f}')
print(f'\n→ Écart-type élevé = l\'algorithme n\'a pas convergé de façon fiable.')

In [ ]:
fig = plot_convergence_runs(results_default,
    title='SA paramètres par défaut (T₀=100, β=0.95)',
    ylabel='Valeur QUBO')
plt.show()

## 3. Impact de la température initiale T₀

T₀ contrôle la phase d'exploration : une T₀ élevée permet d'accepter davantage
de mauvaises solutions au début, favorisant l'exploration globale.
Une T₀ trop faible enferme l'algorithme dans un minimum local dès le départ.

**Règle pratique :** T₀ doit être suffisamment grande pour que le taux d'acceptation
initial soit proche de 1 (≈ exp(-ΔE_moyen / T₀) ≈ 0.8).

In [ ]:
T0_values = [0.1, 1, 10, 50, 100, 500, 1000]
means_T0, stds_T0, times_T0 = [], [], []

for T0 in T0_values:
    vals = []
    t0 = time.time()
    for _ in range(20):
        sa = SimulatedAnnealing(qubo, maximize=False,
                                initial_temperature=T0, cooling_rate=0.95,
                                min_temperature=0.01, steps_per_temperature=1)
        val, _ = sa.solve()
        vals.append(val)
    times_T0.append((time.time() - t0) / 20)
    means_T0.append(np.mean(vals))
    stds_T0.append(np.std(vals))
    print(f'T₀={T0:6.1f}  →  moy={np.mean(vals):8.2f}  std={np.std(vals):6.2f}  temps={times_T0[-1]*1000:.1f}ms')

In [ ]:
fig = plot_parameter_sweep(T0_values, means_T0, stds_T0,
    param_name='Température initiale T₀',
    title='Impact de T₀ sur la qualité (β=0.95, T_min=0.01)',
    ylabel='Valeur QUBO (minimisation)')
plt.xscale('log')
plt.show()

## 4. Impact du facteur de décroissance β (cooling_rate)

β contrôle la vitesse de refroidissement : T_{t+1} = β · T_t.
- β proche de 1 : refroidissement très lent → plus d'exploration, mais plus long
- β proche de 0 : refroidissement rapide → convergence prématurée vers un minimum local

Le nombre total de paliers de température est : ⌊log(T_min/T₀) / log(β)⌋

In [ ]:
beta_values = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99, 0.999]
means_beta, stds_beta, times_beta, n_steps_beta = [], [], [], []

for beta in beta_values:
    n_steps = int(np.floor(np.log(0.01 / 100.0) / np.log(beta)))
    n_steps_beta.append(n_steps)
    vals = []
    t0 = time.time()
    for _ in range(20):
        sa = SimulatedAnnealing(qubo, maximize=False,
                                initial_temperature=100.0, cooling_rate=beta,
                                min_temperature=0.01, steps_per_temperature=1)
        val, _ = sa.solve()
        vals.append(val)
    times_beta.append((time.time() - t0) / 20)
    means_beta.append(np.mean(vals))
    stds_beta.append(np.std(vals))
    print(f'β={beta:.3f}  paliers={n_steps:6d}  →  moy={np.mean(vals):8.2f}  std={np.std(vals):6.2f}  temps={times_beta[-1]*1000:.1f}ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].errorbar(beta_values, means_beta, yerr=stds_beta, fmt='o-', color='steelblue', capsize=4)
axes[0].set_xlabel('Facteur de décroissance β')
axes[0].set_ylabel('Valeur QUBO moyenne')
axes[0].set_title('Qualité vs β (T₀=100, T_min=0.01)')

axes[1].plot(beta_values, [t * 1000 for t in times_beta], 'o-', color='orange')
axes[1].set_xlabel('Facteur de décroissance β')
axes[1].set_ylabel('Temps moyen par run (ms)')
axes[1].set_title('Temps de calcul vs β')

plt.tight_layout()
plt.show()

## 5. Impact du nombre de pas par palier (steps_per_temperature)

`steps_per_temperature` détermine combien de tentatives de mouvement sont effectuées
à chaque niveau de température. Augmenter ce paramètre améliore l'exploration
à chaque palier, au prix d'un temps de calcul supérieur.

C'est une alternative à diminuer β : pour un budget total d'itérations fixé,
on peut soit avoir plus de paliers (β → 1) soit plus de pas par palier.

In [ ]:
steps_values = [1, 2, 5, 10, 20, 50]
means_steps, stds_steps, times_steps = [], [], []

for steps in steps_values:
    vals = []
    t0 = time.time()
    for _ in range(20):
        sa = SimulatedAnnealing(qubo, maximize=False,
                                initial_temperature=100.0, cooling_rate=0.95,
                                min_temperature=0.01, steps_per_temperature=steps)
        val, _ = sa.solve()
        vals.append(val)
    times_steps.append((time.time() - t0) / 20)
    means_steps.append(np.mean(vals))
    stds_steps.append(np.std(vals))
    print(f'steps={steps:3d}  →  moy={np.mean(vals):8.2f}  std={np.std(vals):6.2f}  temps={times_steps[-1]*1000:.1f}ms')

In [ ]:
fig = plot_parameter_sweep(steps_values, means_steps, stds_steps,
    param_name='Nombre de pas par palier',
    title='Impact de steps_per_temperature (T₀=100, β=0.95)',
    ylabel='Valeur QUBO moyenne')
plt.show()

## 6. Comparaison globale : qualité vs temps

Pour un budget de temps donné, quelle combinaison de paramètres donne la meilleure qualité ?

In [ ]:
configs = {
    'T₀=100 β=0.95 s=1'  : dict(initial_temperature=100,  cooling_rate=0.95,  min_temperature=0.01, steps_per_temperature=1),
    'T₀=100 β=0.99 s=1'  : dict(initial_temperature=100,  cooling_rate=0.99,  min_temperature=0.01, steps_per_temperature=1),
    'T₀=500 β=0.95 s=1'  : dict(initial_temperature=500,  cooling_rate=0.95,  min_temperature=0.01, steps_per_temperature=1),
    'T₀=100 β=0.95 s=10' : dict(initial_temperature=100,  cooling_rate=0.95,  min_temperature=0.01, steps_per_temperature=10),
    'T₀=100 β=0.999 s=1' : dict(initial_temperature=100,  cooling_rate=0.999, min_temperature=0.01, steps_per_temperature=1),
}

comparison_data = {}
summary = []
for name, params in configs.items():
    vals = []
    t0 = time.time()
    for _ in range(30):
        sa = SimulatedAnnealing(qubo, maximize=False, **params)
        val, _ = sa.solve()
        vals.append(val)
    elapsed = (time.time() - t0) / 30
    comparison_data[name] = vals
    summary.append((name, np.mean(vals), np.std(vals), elapsed * 1000))
    print(f'{name:25s}  moy={np.mean(vals):8.2f}  std={np.std(vals):6.2f}  temps={elapsed*1000:.1f}ms')

In [ ]:
fig = plot_boxplot_comparison(comparison_data,
    title='Comparaison des configurations de paramètres (n=20)',
    xlabel='Configuration',
    ylabel='Valeur QUBO')
plt.show()

## 7. Analyse critique

### Température initiale T₀
T₀ doit être calibrée par rapport à l'amplitude typique des variations d'énergie ΔE.
Une règle empirique courante (Kirkpatrick et al., 1983) est de choisir T₀ tel que
le taux d'acceptation initial soit ≈ 0.8 : T₀ ≈ -ΔE_moyen / ln(0.8).
Une T₀ trop faible enferme SA dans un minimum local dès le début.
Une T₀ trop grande allonge la phase d'exploration sans gain de qualité.

### Facteur de décroissance β (cooling_rate)
β est le paramètre le plus influent. Un β proche de 1 donne un refroidissement
quasi-adiabatique qui permet à SA de trouver de meilleures solutions, mais au
prix d'un temps de calcul beaucoup plus long (le nombre de paliers croît en
O(1/(1-β))). La littérature recommande β ∈ [0.9, 0.99] comme compromis
(Aarts & Korst, 1989).

### Seuil d'arrêt T_min
T_min n'a d'impact que si l'algorithme atteint ce seuil (vs convergence de la
solution). Diminuer T_min allonge le temps sans améliorer la qualité une fois
que la solution s'est stabilisée.

### Pas par palier (steps_per_temperature)
Augmenter ce paramètre améliore l'exploration à budget total égal (en augmentant
steps et en augmentant β proportionnellement). C'est une alternative à un β lent
qui peut être utile pour des espaces à paysage rugueux.

### Conclusion
Les résultats montrent que la qualité des solutions est très sensible à β et
modérément sensible à T₀. Pour une instance QUBO de taille n=20, les paramètres
recommandés sont : **T₀ ≈ 100–500, β ≈ 0.99, steps_per_temperature ≈ 5–10**.